# 3.2 — Agent with Multiple Tools

A real agent needs a **toolkit** — a set of tools it can choose from.
The model reads each tool's docstring and decides which one to call.

In this notebook we build an agent with 5 tools:
- `calculator` — arithmetic
- `get_current_time` — current date and time
- `get_weather` — mock weather lookup
- `unit_converter` — length/weight conversions
- `search_wikipedia` — topic summaries (mock)

In [ ]:
!pip install langchain langchain-ollama --quiet

## Step 1 — Define the Toolkit

In [ ]:
from langchain_core.tools import tool
from datetime import datetime
import math

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result.
    Examples: '2 + 2', '10 * 5', 'sqrt(144)', '2 ** 8'
    """
    try:
        # Safe eval with only math functions allowed
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        result = eval(expression, {'__builtins__': {}}, allowed)
        return f'{expression} = {result}'
    except Exception as e:
        return f'Error evaluating expression: {e}'

@tool
def get_current_time() -> str:
    """Returns the current date and time."""
    now = datetime.now()
    return now.strftime('Today is %A, %B %d %Y. The time is %H:%M.')

@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    mock_data = {
        'london':   'Cloudy, 14°C, light rain expected',
        'new york': 'Sunny, 22°C, clear skies',
        'tokyo':    'Partly cloudy, 19°C, humid',
        'mumbai':   'Hot and sunny, 35°C',
        'paris':    'Overcast, 16°C',
        'sydney':   'Sunny, 26°C, perfect beach weather',
    }
    return mock_data.get(city.lower(), f'Weather data not available for {city}.')

@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Converts a value between units. Supports: km/miles, kg/lbs, celsius/fahrenheit, meters/feet."""
    conversions = {
        ('km', 'miles'):       lambda x: x * 0.621371,
        ('miles', 'km'):       lambda x: x * 1.60934,
        ('kg', 'lbs'):         lambda x: x * 2.20462,
        ('lbs', 'kg'):         lambda x: x * 0.453592,
        ('celsius', 'fahrenheit'): lambda x: x * 9/5 + 32,
        ('fahrenheit', 'celsius'): lambda x: (x - 32) * 5/9,
        ('meters', 'feet'):    lambda x: x * 3.28084,
        ('feet', 'meters'):    lambda x: x * 0.3048,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f'{value} {from_unit} = {result:.2f} {to_unit}'
    return f'Conversion from {from_unit} to {to_unit} is not supported.'

@tool
def search_wikipedia(topic: str) -> str:
    """Searches for information about a topic. Returns a short summary."""
    mock_wiki = {
        'python':      'Python is a high-level, interpreted programming language known for its clear syntax and readability.',
        'langchain':   'LangChain is a framework for developing applications powered by large language models (LLMs).',
        'rag':         'Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation.',
        'llm':         'A Large Language Model (LLM) is a type of AI model trained on vast text data to generate human-like text.',
        'eiffel tower':'The Eiffel Tower is a wrought-iron lattice tower in Paris, France. Built in 1889, it stands 330 metres tall.',
        'india':       'India is the world\'s largest democracy and second most populous country, located in South Asia.',
    }
    for key, value in mock_wiki.items():
        if key in topic.lower():
            return value
    return f'No summary found for "{topic}".'

# Verify tools
print('Toolkit:')
for t in [calculator, get_current_time, get_weather, unit_converter, search_wikipedia]:
    print(f'  {t.name:<20} — {t.description[:60]}')

## Step 2 — Build the Agent

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

llm = ChatOllama(model='llama3.1', temperature=0)

tools = [calculator, get_current_time, get_weather, unit_converter, search_wikipedia]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are a knowledgeable assistant with access to tools.
Always use tools to get accurate information rather than guessing.
If multiple tools are needed, call them one by one."""

def run_agent(question: str, verbose: bool = True) -> str:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=question)
    ]
    if verbose:
        print(f'Q: {question}')

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            if verbose:
                print(f'A: {response.content}\n')
            return response.content

        for tc in response.tool_calls:
            result = tool_map[tc['name']].invoke(tc['args'])
            if verbose:
                print(f'  [tool] {tc["name"]}({tc["args"]}) → {result}')
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

print('Agent ready.')

## Step 3 — Test Each Tool

In [ ]:
# Each question should trigger a different tool
run_agent('What is the square root of 256?')              # calculator
run_agent('What day is it today?')                        # get_current_time
run_agent('What is the weather like in Tokyo?')           # get_weather
run_agent('Convert 100 km to miles')                      # unit_converter
run_agent('Tell me about LangChain')                      # search_wikipedia

## Step 4 — Multi-Tool Questions

Some questions require the agent to chain multiple tools together.

In [ ]:
# Requires: get_weather + unit_converter
run_agent('What is the weather in London, and what is that temperature in Fahrenheit?')

In [ ]:
# Requires: calculator (twice)
run_agent('What is 2 to the power of 10, and then divide that by 4?')

## Step 5 — Observe Tool Selection

Watch how the model picks the right tool based on the question wording — even with synonyms.

In [ ]:
# Same intent, different phrasing — model should still pick calculator
phrasings = [
    'Compute 45 times 12',
    'How much is 45 multiplied by 12?',
    'Calculate 45 * 12',
]
for q in phrasings:
    run_agent(q)

## Summary

| Concept | Key point |
|---------|----------|
| Tool docstring | The model reads this to decide when to use the tool — write it clearly |
| `bind_tools` | Attaches the tool list to the LLM |
| Tool selection | The model picks based on question semantics, not exact keywords |
| Tool chaining | The model can call multiple tools across loop iterations |
| `tool_map` | Dict for fast tool lookup by name during the loop |